## Part 1 — Setup & launch


In [1]:
# ── 1. GPU check ────────────────────────────────────────────────────────────
# Confirms the runtime has an NVIDIA GPU. Everything still works on CPU, but
# a single sentence can take minutes instead of seconds.
import shutil
import subprocess

if shutil.which("nvidia-smi"):
    print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)
    print("GPU detected — you're good to go.")
else:
    print("=" * 74)
    print("WARNING: no NVIDIA GPU in this runtime.")
    print("Fix: menu bar -> Runtime -> Change runtime type -> T4 GPU,")
    print("then re-run this notebook from the top.")
    print("(Continuing anyway works, but generation will be very slow on CPU.)")
    print("=" * 74)


Fri Oct  9 20:40:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   53C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
# ── 2. Clone + install (idempotent)

import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/debpalash/VoiceStudio.git"
REPO_DIR = "/content/VoiceStudio"
BUN = "/root/.bun/bin/bun"

# Generous network timeouts for package/model downloads.
os.environ.setdefault("UV_HTTP_TIMEOUT", "300")


def run(cmd, *, cwd=None, what=""):
    """Stream command output and stop with an actionable error."""
    shown = cmd if isinstance(cmd, str) else " ".join(map(str, cmd))
    print(f"\n$ {shown}", flush=True)

    result = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        shell=isinstance(cmd, str),
        env=os.environ.copy(),
    )

    if result.returncode != 0:
        raise SystemExit(
            f"\nFAILED: {what or shown} "
            f"(exit code {result.returncode}).\n"
            "Scroll up to inspect the actual error.\n"
            "Fix the error, then rerun this cell."
        )


# ── 2a. Clone repository, or reuse the existing checkout ──────────────────

repo_path = Path(REPO_DIR)

if (repo_path / ".git").is_dir():
    print(f"Repository already exists at {REPO_DIR} — reusing it.")
elif repo_path.exists():
    raise SystemExit(
        f"{REPO_DIR} exists but is not a Git repository.\n"
        "Rename/remove that directory, then rerun this cell."
    )
else:
    run(
        ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
        what="Git clone",
    )


# ── 2b. System dependencies ───────────────────────────────────────────────

run(
    "apt-get -qq update && "
    "apt-get -qq install -y ffmpeg libsndfile1",
    what="Install system packages",
)


# ── 2c. Install Bun ───────────────────────────────────────────────────────

if not os.path.isfile(BUN):
    run(
        "curl -fsSL https://bun.sh/install | bash",
        what="Install Bun",
    )

if not os.path.isfile(BUN):
    raise SystemExit(
        f"Bun executable was not found at {BUN} after installation."
    )

os.environ["PATH"] = (
    os.path.dirname(BUN) + os.pathsep + os.environ.get("PATH", "")
)

run([BUN, "--version"], what="Verify Bun")


# ── 2d. Build the web frontend (FIXED) ────────────────────────────────────
#
# IMPORTANT:
# The current repo does not use `bun run --cwd frontend build`.
# Its root package.json defines `build:web`, which invokes the
# web-specific Vite build in electron/.
#
# Vite writes the generated web UI to:
# /content/VoiceStudio/frontend/dist/index.html

dist_index = Path(REPO_DIR) / "frontend" / "dist" / "index.html"

if dist_index.is_file():
    print(f"Web frontend already built: {dist_index}")
else:
    # Install workspace dependencies from the repository root.
    run(
        [BUN, "install", "--frozen-lockfile"],
        cwd=REPO_DIR,
        what="Install Bun workspace dependencies",
    )

    # FIX: invoke the actual root-level web build script.
    run(
        [BUN, "run", "build:web"],
        cwd=REPO_DIR,
        what="Build VoiceStudio web frontend",
    )

    if not dist_index.is_file():
        raise SystemExit(
            "\nThe build command returned successfully, but the expected "
            "web entry point is missing:\n"
            f"  {dist_index}\n\n"
            "Inspect the build output above. The current web build is "
            "configured in electron/vite.web.config.ts."
        )

print(f"✅ Web frontend ready: {dist_index}")


# ── 2e. Install Python backend dependencies ───────────────────────────────

if not shutil.which("uv"):
    run(
        [sys.executable, "-m", "pip", "install", "-q", "uv"],
        what="Install uv",
    )

UV = shutil.which("uv")

if not UV:
    raise SystemExit(
        "uv was installed but could not be found on PATH. "
        "Restart the runtime or check the installation."
    )

run(
    [
        UV,
        "pip",
        "install",
        "--system",
        "--no-cache",
        "--constraint",
        "deploy/torch-constraints.txt",
        ".",
    ],
    cwd=REPO_DIR,
    what="Install VoiceStudio Python backend",
)


# ── 2f. Optional cuDNN 8 compatibility setup for WhisperX ─────────────────

# scripts/setup.py expects this directory to exist. We use Colab's system
# Python directly; this is NOT creating or activating a virtual environment.

os.makedirs(os.path.join(REPO_DIR, ".venv"), exist_ok=True)

run(
    [
        sys.executable,
        os.path.join(REPO_DIR, "scripts", "setup.py"),
    ],
    cwd=REPO_DIR,
    what="cuDNN 8 compatibility setup",
)


# ── 2g. Verify the installed model stack in a fresh interpreter ───────────

sanity_script = (
    "import torch, torchaudio, uvicorn, fastapi, transformers; "
    "print(f'torch {torch.__version__}, "
    "torchaudio {torchaudio.__version__}, "
    "transformers {transformers.__version__}, "
    "CUDA available: {torch.cuda.is_available()}'); "
    "from omnivoice.models.omnivoice import OmniVoice; "
    "from transformers import HiggsAudioV2TokenizerModel; "
    "print('Install OK - backend model stack imports cleanly')"
)

run(
    [sys.executable, "-c", sanity_script],
    cwd=REPO_DIR,
    what="Python and model-stack import sanity check",
)

print("\n✅ Cell 2 completed successfully.")


$ git clone --depth 1 https://github.com/debpalash/VoiceStudio /content/VoiceStudio

$ apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1

$ curl -fsSL https://bun.sh/install | bash

$ /root/.bun/bin/bun --version

$ /root/.bun/bin/bun install --frozen-lockfile

$ /root/.bun/bin/bun run --cwd frontend build


SystemExit: The frontend build finished but frontend/dist/index.html is missing —
scroll up for vite errors, then re-run this cell.

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)


In [ ]:
# ── 3. (Optional) Hugging Face token ────────────────────────────────────────
# Only needed for GATED models — e.g. pyannote speaker diarization, used by
# video dubbing for multi-speaker detection. TTS, voice cloning, and voice
# design all work WITHOUT a token, so feel free to skip this cell.
#
# To use one: accept the model terms at
#   https://huggingface.co/pyannote/speaker-diarization-3.1
# then add a Colab Secret named HF_TOKEN (key icon in the left sidebar),
# toggle "Notebook access" ON, and re-run this cell.
import os

try:
    from google.colab import userdata
    _token = userdata.get("HF_TOKEN")
    os.environ["HF_TOKEN"] = _token
    print("HF_TOKEN loaded from Colab Secrets — gated models (diarization) enabled.")
except Exception:
    print("No HF_TOKEN Colab Secret found — skipping. (Everything except gated "
          "diarization models works without it.)")


In [ ]:
# ── 4. (Optional, recommended) Pre-download the default voice model ─────────
# The default TTS engine fetches k2-fsa/OmniVoice (a few GB) on its very first
# generation. Downloading it up front makes the first request fast instead of
# a several-minute stall. Safe to re-run: already-downloaded files are reused.
from huggingface_hub import snapshot_download

path = snapshot_download("k2-fsa/OmniVoice")
print("Default TTS model cached at:", path)


In [ ]:
# ── 5. Launch the backend ───────────────────────────────────────────────────
# Starts uvicorn on port 3900 and waits for /health. Re-running this cell when
# the backend is already up just reports its status.
import json
import os
import subprocess
import sys
import time
import urllib.request

REPO_DIR = "/content/VoiceStudio"
PORT = 3900
LOG_PATH = "/content/omnivoice_backend.log"
HEALTH_URL = f"http://127.0.0.1:{PORT}/health"

def health():
    try:
        with urllib.request.urlopen(HEALTH_URL, timeout=5) as r:
            return json.load(r)
    except Exception:
        return None

info = health()
if info:
    print(f"Backend already running — {info}")
else:
    # Clear any half-dead process from a previous run of this cell.
    subprocess.run(f"kill -9 $(lsof -t -i:{PORT}) 2>/dev/null || true", shell=True)
    time.sleep(1)

    env = os.environ.copy()
    # Headless-server deployment, same as the official Docker image: relaxes
    # the desktop-only loopback origin gate so the proxied browser session can
    # reach the settings/system routes.
    env["OMNIVOICE_SERVER_MODE"] = "1"
    # Voices, projects, and generated audio live here. Ephemeral! See the
    # troubleshooting cell for persisting it to Google Drive.
    env["OMNIVOICE_DATA_DIR"] = "/content/omnivoice_data"
    env["PYTHONUNBUFFERED"] = "1"

    log = open(LOG_PATH, "ab")
    proc = subprocess.Popen(
        [sys.executable, "-m", "uvicorn", "main:app",
         "--app-dir", "backend", "--host", "127.0.0.1", "--port", str(PORT)],
        cwd=REPO_DIR, env=env, stdout=log, stderr=subprocess.STDOUT,
    )
    print(f"Backend starting (PID {proc.pid}); log: {LOG_PATH}")

    deadline = time.time() + 300
    while time.time() < deadline:
        if proc.poll() is not None:
            break  # process died — report below
        info = health()
        if info:
            break
        print(".", end="", flush=True)
        time.sleep(3)
    print()

    if info:
        print(f"Backend is up — {info}")
        if "cuda" not in str(info.get("device", "")):
            print("NOTE: device is not CUDA — generation will be slow. "
                  "See cell 1 to enable the GPU runtime.")
    else:
        try:
            with open(LOG_PATH, "r", errors="replace") as f:
                tail = "".join(f.readlines()[-40:])
        except OSError:
            tail = "(no log file found)"
        raise SystemExit(
            "Backend did not become healthy within 5 minutes.\n"
            f"--- last lines of {LOG_PATH} ---\n{tail}\n"
            "--- end of log ---\n"
            "Fix the error above (usually a missing dependency: re-run cell 2),\n"
            "then re-run this cell. Still stuck? Attach the log to an issue:\n"
            "  https://github.com/debpalash/VoiceStudio/issues"
        )


In [ ]:
# ── 6. Open the app ─────────────────────────────────────────────────────────
# Serves localhost:3900 to YOUR browser through Colab's built-in kernel port
# proxy — authenticated to your Google session, no third-party tunnel binary.
# A new browser tab opens with the full VoiceStudio UI (allow pop-ups
# for colab.research.google.com if nothing appears).
#
# Want a PUBLIC URL instead (e.g. to open the app on your phone)? Cloudflare's
# quick tunnel works well — run in a new cell:
#   !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
#   !chmod +x /usr/local/bin/cloudflared
#   !nohup cloudflared tunnel --url http://127.0.0.1:3900 --no-autoupdate > /content/cloudflared.log 2>&1 &
#   !sleep 5 && grep -o "https://.*trycloudflare.com" /content/cloudflared.log | head -1
# Anyone with that URL can reach your session — set a share PIN in the app's
# Settings first.
from google.colab import output

output.serve_kernel_port_as_window(3900)
print("If no tab opened, click the link above (and allow pop-ups).")


In [ ]:
# ── 7. Smoke test: generate speech through the API ──────────────────────────
# Proves the whole stack end-to-end without touching the UI: /health, then a
# real TTS generation played inline. POST /generate returns the WAV directly.
import json
import urllib.error
import urllib.parse
import urllib.request

BASE = "http://127.0.0.1:3900"

with urllib.request.urlopen(f"{BASE}/health", timeout=10) as r:
    print("GET /health ->", json.load(r))

data = urllib.parse.urlencode({
    "text": "VoiceStudio is up and running on Google Colab.",
}).encode()
print("Generating... (first-ever generation loads the model — if you skipped "
      "cell 4 it also downloads it, which can take several minutes)")
try:
    with urllib.request.urlopen(
        urllib.request.Request(f"{BASE}/generate", data=data), timeout=1800
    ) as r:
        wav = r.read()
        print(f"OK — {len(wav)} bytes, duration {r.headers.get('X-Audio-Duration')}s, "
              f"generated in {r.headers.get('X-Gen-Time')}s (seed {r.headers.get('X-Seed')})")
except urllib.error.HTTPError as e:
    detail = e.read().decode("utf-8", errors="replace")
    raise SystemExit(
        f"Generation failed: HTTP {e.code}\n{detail}\n"
        "Check /content/omnivoice_backend.log for the full trace; on a fresh\n"
        "session the usual cause is an interrupted model download — re-run\n"
        "cell 4, then this cell."
    )

OUT = "/content/omnivoice_smoke.wav"
with open(OUT, "wb") as f:
    f.write(wav)

from IPython.display import Audio, display
display(Audio(OUT))
